# 158 — Layer Bypass Analysis

Transformers have skip connections at every layer. Some inputs may effectively
bypass certain layers, routing information directly through the residual stream.
This module detects bypass patterns, measures effective depth, and finds
prediction shortcuts.

## Why This Matters

Layer bypass characterizes what each layer contributes to the model's computation. Understanding layer-level organization — which layers are critical, which are redundant, and how they specialize — is essential for both interpretability and efficient model design.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.layer_bypass_analysis import (
    layer_contribution_vs_passthrough,
    effective_depth,
    skip_connection_utilization,
    shortcut_detection,
    minimal_circuit_depth,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.arange(15)

## Contribution vs Pass-through

What fraction of each layer's output is new computation vs relayed input?

In [ ]:
result = layer_contribution_vs_passthrough(model, tokens)
print(f"Bypass layers: {result['bypass_layers']}\n")
for p in result['per_layer']:
    tag = ' <-- BYPASS' if p['is_bypass'] else ''
    print(f"Layer {p['layer']}: pass={p['passthrough_ratio']:.3f}  "
          f"new={p['contribution_ratio']:.3f}{tag}")

## Effective Depth

How many layers are actually needed for this input?

In [ ]:
result = effective_depth(model, tokens)
print(f"Effective depth: {result['effective_depth']}/{result['total_layers']}")
print(f"Depth ratio: {result['depth_ratio']:.2f}")
print(f"Active layers: {result['active_layers']}\n")
for p in result['per_layer']:
    bar = '#' * int(p['relative_contribution'] * 30)
    print(f"Layer {p['layer']}: rel_contrib={p['relative_contribution']:.4f}  {bar}")

## Skip Connection Utilization

How much does the embedding survive to the final layer?

In [ ]:
result = skip_connection_utilization(model, tokens)
print(f"Embedding-final cosine: {result['embedding_final_cosine']:.4f}")
print(f"Retained fraction: {result['embedding_retained_fraction']:.4f}")
print(f"Embedding norm: {result['embedding_norm']:.4f}")
print(f"Final norm: {result['final_norm']:.4f}")
print(f"\nPer-layer contributions: {result['per_layer_contribution']}")

## Shortcut Detection

Can the model already predict the answer at an early layer?

In [ ]:
result = shortcut_detection(model, tokens)
print(f"Target token: {result['target_token']}")
print(f"Has shortcut: {result['has_shortcut']}")
print(f"Earliest correct: layer {result['earliest_correct_layer']}\n")
for p in result['per_layer']:
    tag = ' *' if p['correct'] else ''
    print(f"Layer {p['layer']}: pred={p['top_prediction']:3d}  "
          f"target_prob={p['target_probability']:.4f}{tag}")

## Minimal Circuit Depth

How many layers does it take to achieve 90% of the final logit?

In [ ]:
result = minimal_circuit_depth(model, tokens, logit_threshold=0.9)
print(f"Minimal depth: {result['minimal_depth']}/{result['total_layers']}")
print(f"Final logit: {result['final_logit']:.4f}\n")
for p in result['per_layer']:
    bar = '#' * int(abs(p['fraction_of_final']) * 30)
    print(f"Layer {p['layer']}: logit={p['logit']:+.4f}  "
          f"fraction={p['fraction_of_final']:.3f}  {bar}")